# SOV3³ Small-Model Runner + Governance-NN Trainer (Kaggle)_Generated 2026-07-07 for MEOK/SOV3³._**Purpose:** use Kaggle's free **2×T4 (32GB combined)** to run the estate's *small* brainsand (re)train a governance neural net — the work that does NOT need the 192GB M4 or a paid A100.**Before running:** Notebook → Settings → **Accelerator = GPU T4 x2**, **Internet = On**.**Scope (honest):**- ✅ Runs qwen2.5:3b / llama3.2:3b / deepseek-r1:7b / BGE-M3 embeddings via Ollama.- ✅ Retrains an sklearn governance NN (care/threat/relationship) on more data.- ❌ Does NOT run qwen3:30b-a3b — 30B won't fit T4 comfortably; use Modal/A100 for that.

## 1. Install Ollama + start server (GPU-backed)

In [ ]:
import subprocess, time, os, requests# Install Ollama (Linux)subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=False)# Start server in backgroundsrv = subprocess.Popen(["ollama","serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)for _ in range(30):    try:        if requests.get("http://127.0.0.1:11434/api/tags", timeout=2).ok:            print("ollama up"); break    except Exception: time.sleep(1)print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout)

## 2. Pull the small estate models

In [ ]:
for m in ["qwen2.5:3b", "llama3.2:3b", "deepseek-r1:7b"]:    print("pulling", m, "...")    subprocess.run(["ollama","pull",m], check=False)print(requests.get("http://127.0.0.1:11434/api/tags").json())

## 3. SOV3³ four-brain routing demo (small-model tier)Routes a query to a brain persona. This mirrors the SOV3³ router shape; swap in the 30B anchor when on A100.

In [ ]:
import requests, jsonBRAINS = {  "COMPLIANCE": ("qwen2.5:3b", "You are SOVEREIGN-COMPLIANCE. Answer with EU AI Act / governance framing. Be precise, cite obligations."),  "DEFENSE":    ("deepseek-r1:7b", "You are SOVEREIGN-DEFENSE. Reason step by step about threats and safeguards."),  "INTUITION":  ("qwen2.5:3b", "You are SOVEREIGN-INTUITION. Answer with pattern-level, care-first insight."),  "VOICE":      ("llama3.2:3b", "You are SOVEREIGN-VOICE. Answer warmly and concisely for a human."),}def route(brain, prompt):    m, sys = BRAINS[brain]    r = requests.post("http://127.0.0.1:11434/api/generate",        json={"model":m,"system":sys,"prompt":prompt,"stream":False}, timeout=300)    return r.json().get("response","")print("VOICE:", route("VOICE", "In one sentence, what is a sovereign AI OS?"))print("\nCOMPLIANCE:", route("COMPLIANCE", "Name one EU AI Act obligation for a high-risk system.")[:400])

## 4. BGE-M3 embeddings (the estate's embedding model)

In [ ]:
from sentence_transformers import SentenceTransformeremb = SentenceTransformer("BAAI/bge-m3", device="cuda")v = emb.encode(["sovereign governance", "care-first AI", "compliance passport"], normalize_embeddings=True)print("embedding shape:", v.shape)

## 5. Retrain governance NNs on LIVE episode data (BGE-M3 GPU features)Uses the **real** exported episodes (`sov3_governance_episodes.csv`, 1,076 rows across 8 NNs)— upload it as a Kaggle Dataset first (it attaches under `/kaggle/input/`). Features =BGE-M3 GPU embeddings of each episode's `content`; target = `care_weight`. This is thesame pipeline validated locally (relationship MAE 0.142 / threat 0.483 on TF-IDF; BGE-M3should improve it).

In [ ]:
import pandas as pd, numpy as np, glob, joblibfrom sklearn.neural_network import MLPRegressorfrom sklearn.model_selection import cross_val_score# locate the uploaded dataset (Kaggle mounts datasets under /kaggle/input/<slug>/)hits = glob.glob("/kaggle/input/**/sov3_governance_episodes.csv", recursive=True)assert hits, "Upload sov3_governance_episodes.csv as a Kaggle Dataset and attach it (Add Input)."df = pd.read_csv(hits[0]).dropna(subset=["content"])df["care_weight"] = df["care_weight"].fillna(df["importance_score"]).fillna(0.5)print("loaded", len(df), "episodes across", df["nn"].nunique(), "NNs")# BGE-M3 GPU embeddings (the estate's real embedding model) — reuse `emb` from cell 4if "emb" not in dir():    from sentence_transformers import SentenceTransformer    emb = SentenceTransformer("BAAI/bge-m3", device="cuda")results = {}for nn, g in df.groupby("nn"):    if len(g) < 20:        results[nn] = f"skip (n={len(g)})"; continue    X = emb.encode(g["content"].str[:2000].tolist(), normalize_embeddings=True,                   batch_size=32, show_progress_bar=False)    y = g["care_weight"].to_numpy()    model = MLPRegressor(hidden_layer_sizes=(128,64), max_iter=800,                         early_stopping=True, random_state=0)    cv = min(5, max(2, len(g)//30))    mae = -cross_val_score(model, X, y, cv=cv, scoring="neg_mean_absolute_error").mean()    model.fit(X, y)    joblib.dump(model, f"/kaggle/working/{nn}_nn_bge.joblib")    results[nn] = f"MAE {mae:.3f} (n={len(g)}, saved)"for nn, r in sorted(results.items()):    print(f"{nn:<14} {r}")

## 6. Next steps- Export real feature/label logs from `sovereign-temple/neural_core` → upload as a Kaggle Dataset → point cell 5 at it to fix the 4 weak NNs.- For the **30B anchor**, use Modal (per-second A100/H100) — see `SOV3_FREE_COMPUTE_SURVEY_2026-07-07.md`.- Kaggle background execution: commit + "Save & Run All" so training continues after you close the tab (~9h/session, ~30h/week).